# 04c · FT-Transformer vs. XGBoost: Head-to-Head

This notebook deliberately never imports `torch`. Running PyTorch and XGBoost's native
code in the same process crashed the development machine with a segfault -- two separate
copies of the OpenMP runtime (one bundled with the PyTorch wheel, one pulled in via
scikit-learn/XGBoost's Homebrew-linked OpenMP) ended up loaded together, and mixing them
is a known-unsafe class of bug on macOS (LLVM's own OpenMP documentation says the usual
`KMP_DUPLICATE_LIB_OK=TRUE` workaround only suppresses the startup abort -- it explicitly
does *not* claim to fix the underlying conflict, which "may cause incorrect results or
crashes"). See [`04b_neural_network_benchmark.ipynb`](04b_neural_network_benchmark.ipynb)
for the full crash writeup.

**The fix is structural, not a flag**: notebook 04b trains the FT-Transformer and saves its
results to `../reports/ft_transformer_results.json`, never importing `xgboost`. This
notebook loads that JSON and benchmarks the deployed XGBoost model on its own, never
importing `torch`. The two libraries' native code never executes in the same OS process,
so the conflict can't occur, by construction -- rather than relying on an environment
variable that isn't actually guaranteed to be safe.

In [1]:
import json
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score

with open("../data/processed/feature_columns.json") as f:
    cfg = json.load(f)
FEATURES, LABEL = cfg["features"], cfg["label"]

test = pd.read_parquet("../data/processed/test.parquet")
y_test_arr = test[LABEL].values.astype(np.float32)

with open("../reports/ft_transformer_results.json") as f:
    ft_results = json.load(f)

print("Loaded FT-Transformer results from notebook 04b:")
print(json.dumps(ft_results, indent=2))

Loaded FT-Transformer results from notebook 04b:
{
  "test_roc_auc": 0.9868001450263408,
  "test_pr_auc": 0.7089066919746058,
  "inference_latency_ms_median": 0.5971460004730034,
  "n_params": 72897,
  "train_time_sec": 1004.6460661888123,
  "epochs_trained": 13,
  "train_sample_size": 120000,
  "val_sample_size": 20000,
  "device": "cpu",
  "mlflow_run_id": "bb69a9b462b14a648f48e96a8641e3ce"
}


## Benchmark the deployed XGBoost model

Loads the exact artifact `src/api/` serves (`models/fraud_xgboost.json`) and evaluates it
on the same held-out test set, with the same single-transaction CPU latency methodology
notebook 04b used for the FT-Transformer -- this is how the serving API actually calls a
model (`src/api/main.py` scores one incoming transaction at a time), so a batched
throughput number would be misleading for this comparison.

In [2]:
xgb_model = xgb.XGBClassifier()
xgb_model.load_model("../models/fraud_xgboost.json")

xgb_test_proba = xgb_model.predict_proba(test[FEATURES])[:, 1]
xgb_test_roc = roc_auc_score(y_test_arr, xgb_test_proba)
xgb_test_pr = average_precision_score(y_test_arr, xgb_test_proba)

xgb_model.predict_proba(test[FEATURES].iloc[:1])  # warmup
xgb_latencies = []
for i in range(200):
    t0 = time.perf_counter()
    xgb_model.predict_proba(test[FEATURES].iloc[i:i + 1])
    xgb_latencies.append((time.perf_counter() - t0) * 1000)
xgb_latency_ms = float(np.median(xgb_latencies))

xgb_n_tree_nodes = int(xgb_model.get_booster().trees_to_dataframe().shape[0])

print(f"XGBoost  test ROC-AUC={xgb_test_roc:.4f}  test PR-AUC={xgb_test_pr:.4f}")
print(f"XGBoost single-transaction CPU latency (median of 200): {xgb_latency_ms:.2f}ms")

XGBoost  test ROC-AUC=0.9903  test PR-AUC=0.7220
XGBoost single-transaction CPU latency (median of 200): 5.71ms


## Head-to-head comparison and verdict

In [3]:
comparison = pd.DataFrame([
    {
        "model": "xgboost (deployed)",
        "test_roc_auc": xgb_test_roc,
        "test_pr_auc": xgb_test_pr,
        "inference_latency_ms_median": xgb_latency_ms,
        "params_or_tree_nodes": xgb_n_tree_nodes,
    },
    {
        "model": "ft_transformer",
        "test_roc_auc": ft_results["test_roc_auc"],
        "test_pr_auc": ft_results["test_pr_auc"],
        "inference_latency_ms_median": ft_results["inference_latency_ms_median"],
        "params_or_tree_nodes": ft_results["n_params"],
    },
]).set_index("model")
comparison

,test_roc_auc,test_pr_auc,inference_latency_ms_median,params_or_tree_nodes
model,,,,
xgboost (deployed),0.990348,0.721958,5.705666,87198
ft_transformer,0.986800,0.708907,0.597146,72897


In [4]:
pr_auc_delta = ft_results["test_pr_auc"] - xgb_test_pr
roc_auc_delta = ft_results["test_roc_auc"] - xgb_test_roc
latency_ratio = ft_results["inference_latency_ms_median"] / xgb_latency_ms

if pr_auc_delta > 0:
    verdict = (
        f"FT-Transformer scores higher on both ranking metrics on this held-out test set "
        f"(PR-AUC {pr_auc_delta:+.4f}, ROC-AUC {roc_auc_delta:+.4f}), trained on a "
        f"{ft_results['train_sample_size']:,}-row compute-budget subsample in "
        f"{ft_results['train_time_sec']:.0f}s on CPU. XGBoost stays the deployed model "
        f"anyway: the entire serving stack (src/api/model.py), SHAP explainability "
        f"(notebook 05's TreeExplainer), and drift monitoring (notebook 06) are all built "
        f"around xgb.XGBClassifier's native format. Shipping the transformer would mean "
        f"re-deriving explainability (Captum/integrated-gradients instead of TreeExplainer) "
        f"and re-validating the whole threshold-calibration pipeline against a differently "
        f"-shaped score distribution -- a real migration, not a drop-in swap, and not "
        f"justified by this gap alone given XGBoost's already-strong numbers. The gap is "
        f"also partly confounded: the FT-Transformer sees the raw `category` field as a "
        f"learned embedding, which XGBoost never sees directly (only its smoothed target "
        f"encoding) -- see notebook 04b's opening note."
    )
else:
    verdict = (
        f"XGBoost still wins or ties on this held-out test set (PR-AUC {-pr_auc_delta:.4f} "
        f"higher, ROC-AUC {-roc_auc_delta:.4f} higher) despite the FT-Transformer having a "
        f"native-categorical-embedding advantage XGBoost doesn't get (see notebook 04b's "
        f"opening note). This is consistent with Grinsztajn et al. 2022's finding that tree "
        f"ensembles remain the stronger default on structured tabular data with irregular "
        f"feature interactions (the velocity/z-score features here) even against a modern "
        f"attention-based architecture. XGBoost stays deployed."
    )

print(f"PR-AUC:  XGBoost={xgb_test_pr:.4f}  FT-Transformer={ft_results['test_pr_auc']:.4f}  "
      f"(delta {pr_auc_delta:+.4f})")
print(f"ROC-AUC: XGBoost={xgb_test_roc:.4f}  FT-Transformer={ft_results['test_roc_auc']:.4f}  "
      f"(delta {roc_auc_delta:+.4f})")
print(f"Inference latency: XGBoost={xgb_latency_ms:.2f}ms  "
      f"FT-Transformer={ft_results['inference_latency_ms_median']:.2f}ms  "
      f"({latency_ratio:.1f}x)")
print()
print(verdict)

PR-AUC:  XGBoost=0.7220  FT-Transformer=0.7089  (delta -0.0131)
ROC-AUC: XGBoost=0.9903  FT-Transformer=0.9868  (delta -0.0035)
Inference latency: XGBoost=5.71ms  FT-Transformer=0.60ms  (0.1x)

XGBoost still wins or ties on this held-out test set (PR-AUC 0.0131 higher, ROC-AUC 0.0035 higher) despite the FT-Transformer having a native-categorical-embedding advantage XGBoost doesn't get (see notebook 04b's opening note). This is consistent with Grinsztajn et al. 2022's finding that tree ensembles remain the stronger default on structured tabular data with irregular feature interactions (the velocity/z-score features here) even against a modern attention-based architecture. XGBoost stays deployed.


In [5]:
benchmark_summary = {
    "xgboost": {
        "test_roc_auc": float(xgb_test_roc),
        "test_pr_auc": float(xgb_test_pr),
        "inference_latency_ms_median": xgb_latency_ms,
        "tree_nodes": xgb_n_tree_nodes,
    },
    "ft_transformer": ft_results,
    "deployed_model": "xgboost",
    "verdict": verdict,
}
with open("../reports/nn_benchmark.json", "w") as f:
    json.dump(benchmark_summary, f, indent=2)

print("Saved ../reports/nn_benchmark.json")

Saved ../reports/nn_benchmark.json
